# nanoGPT: pre-train on CodeParrot, fine-tune a classifier on xCodeEval (compiles / compile error)

Edit **only the config cell** between runs.

**Pipeline**
1. Pre-train a small GPT (next-token prediction) on CodeParrot Python files.
2. Build a labelled set from xCodeEval: Python submissions that compile (label 1) or give a compilation error (label 0).
3. Put a classification head on top of the GPT and train it on the labelled set (mean-pool the token states -> head -> 2 classes).
4. Compare how well this works for different head depths, different numbers of frozen layers, and different amounts of labelled data.

## What is being compared

Every fine-tuning run is one point in this grid:

| Axis | Values | Meaning |
|---|---|---|
| **Initialisation** | pre-trained / scratch | Start from the CodeParrot-pre-trained GPT, or from random weights (the "does pre-training help at all?" baseline) |
| **Frozen blocks** (`n_freeze`) | `0`, `N_LAYER//4`, `N_LAYER//2`, `N_LAYER` | How many of the *bottom* transformer blocks keep their pre-trained weights fixed. `0` = full fine-tune. `N_LAYER` = fully frozen GPT, only the head learns (a "linear probe" when the head has depth 1). Whenever `n_freeze > 0` the token (and position) embeddings are frozen as well |
| **Head depth** | 1, 2, 3 | 1 = a single linear layer. 2 = linear -> GELU -> linear. 3 = linear -> GELU -> linear -> GELU -> linear (hidden width `HEAD_HIDDEN`) |
| **Labelled examples** | 500, 1000, 2500, 5000 | Class-balanced subsets of the labelled training pool (data-efficiency curve) |
| **Seed** | `GRID_SEEDS` | Repeats to separate real differences from noise |

The scratch baseline uses the same head depths and label sizes but is always fully trainable (`n_freeze = 0`).

Separate axes that need their own pre-training run: positional encoding (`POS_ENCS` in the config cell, e.g. `("rope", "abs")`; the notebook runs all of them in turn, each with its own checkpoint folder and `grid_results_<pos_enc>.csv`) and the amount of pre-training (`PRETRAIN_FILES`, `PRETRAIN_ITERS`).

In [2]:
# ======================================================================
# CONFIG - the only cell you should need to edit between runs
# ======================================================================
import os, torch

PRESET  = "gpu"     # "cpu" = the small laptop run you did before | "gpu" = bigger run for a cloud GPU
POS_ENCS = ("rope", "abs")   # positional encodings to compare; each is pre-trained and evaluated in turn.
                             # options: "rope" = rotary, "abs" = GPT-2 learned absolute, "nope" = none
POS_ENC  = POS_ENCS[0]       # used by the single-run section; the pre-training and grid loops switch it automatically
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True   # faster fp32 matmuls on Ampere+ GPUs (fine-tuning runs in fp32)
    torch.backends.cudnn.allow_tf32 = True

# ---- Size of the experiment (chosen by PRESET) ----
# BLOCK_SIZE is also the max length of an xCodeEval program (longer ones are dropped when building labels),
# so if you change it, delete labels_xcodeeval.json and rebuild the labels.
if PRESET == "cpu":
    PRETRAIN_FILES, PRETRAIN_ITERS = 5000, 3000        # ~9M tokens seen, <1 pass over the data
    BATCH_SIZE, BLOCK_SIZE, GRAD_ACCUM = 12, 256, 1
    N_LAYER, N_HEAD, N_EMBD = 4, 4, 128                # ~7M parameters
    XC_MAX_SHARDS = 8
    GRID_SEEDS = (0,)
elif PRESET == "gpu":
    PRETRAIN_FILES, PRETRAIN_ITERS = 20_000, 10_000    # ~180M tokens of data, ~330M tokens seen (~2 passes)
    BATCH_SIZE, BLOCK_SIZE, GRAD_ACCUM = 64, 256, 2    # 16,384 tokens per iteration
    N_LAYER, N_HEAD, N_EMBD = 8, 8, 512                # ~50M parameters (about half is the GPT-2 embedding table)
    XC_MAX_SHARDS = 400                                 # more shards -> enough compile-error examples (the rare class)
    GRID_SEEDS = (0, 1, 2)                             # drop to (0,) for a quick first pass

    # PRETRAIN_FILES, PRETRAIN_ITERS = 500, 300          # SMOKE TEST (full: 40_000, 20_000)
    # BATCH_SIZE, BLOCK_SIZE, GRAD_ACCUM = 32, 256, 2    # 16,384 tokens per iteration
    # N_LAYER, N_HEAD, N_EMBD = 8, 8, 512                # ~50M parameters
    # XC_MAX_SHARDS = 40                                 # SMOKE TEST (full: 40)
    # GRID_SEEDS = (0,)                                  # SMOKE TEST (full: (0, 1, 2), or (0,) for a first pass)
else:
    raise ValueError("PRESET must be 'cpu' or 'gpu'")

# ---- xCodeEval labelled data (built once, see "Build labels") ----
XC_LANG_CLUSTER = "Python"
XC_N_PER_CLASS  = 5000      # stop collecting once we have this many examples per class
LABEL_FILE      = "labels_xcodeeval.json"

# ---- Fine-tuning (classifier), used by the "single runs" section ----
HEAD_DEPTH  = 1             # number of linear layers in the head: 1, 2 or 3
HEAD_HIDDEN = 256           # hidden width of the head (ignored when HEAD_DEPTH = 1)
N_LABELED   = 4800          # labelled training examples used (None = whole pool), class-balanced
FT_EPOCHS   = 10
N_FREEZE    = 0             # bottom blocks to freeze: 0 = full fine-tune, N_LAYER = whole GPT frozen
LR_HEAD     = 1e-3          # learning rate of the (randomly initialised) head
LR_BACKBONE = 1e-4          # learning rate of the pre-trained GPT blocks that are not frozen
LR_BACKBONE_SCRATCH = 1e-4  # same, but for the random-init baseline (it may want a higher LR than 1e-4)

# ---- Experiment grid (section 7) ----
GRID_FREEZE = sorted({0, N_LAYER // 4, N_LAYER // 2, N_LAYER})   # e.g. 8 layers -> 0, 2, 4, 8
GRID_DEPTHS = (1, 2, 3)
GRID_SIZES  = (500, 1000, 2500, 5000)
GRID_EPOCHS = 5

# GRID_FREEZE = sorted({0, N_LAYER // 4, N_LAYER // 2, N_LAYER})   # 8 layers -> 0, 2, 4, 8
# GRID_DEPTHS = (1,)                 # SMOKE TEST (full: (1, 2, 3))
# GRID_SIZES  = (500,)               # SMOKE TEST (full: (500, 1000, 2500, 5000))
# GRID_EPOCHS = 5
RESULTS_CSV = f"grid_results_{POS_ENC}.csv"                                  # grid results are appended here as they finish

REUSE_PRETRAINED = False   # True: skip pre-training for an encoding whose checkpoint already exists (e.g. after an interrupted run)
def pretrain_dir(pe):      # where train.py writes ckpt.pt for positional encoding `pe`
    return f"out-code-{pe}-{PRETRAIN_ITERS}it-{PRETRAIN_FILES}f"
PRETRAIN_DIR = pretrain_dir(POS_ENC)
tok_per_iter = BATCH_SIZE * BLOCK_SIZE * GRAD_ACCUM
print(f"device={DEVICE} preset={PRESET} positional encodings={POS_ENCS}")
print(f"{tok_per_iter:,} tokens/iter -> {tok_per_iter*PRETRAIN_ITERS/1e6:.1f}M tokens seen in pre-training")

device=cuda preset=gpu positional encodings=('rope', 'abs')
32,768 tokens/iter -> 327.7M tokens seen in pre-training


## 1. Build labels from xCodeEval (run once; label 1 = compiles, 0 = compilation error)

Runs first so you know early how many compile-error examples there are. Compile errors are rare (roughly 20-50 per shard), so this needs many shards.

In [3]:
import os, json, random, time, tiktoken
from huggingface_hub import list_repo_files, hf_hub_download

enc = tiktoken.get_encoding("gpt2")      # same tokenizer as the rest of the notebook (needed here for the length filter)

if os.path.exists(LABEL_FILE):
    print(f"{LABEL_FILE} already exists - reusing it (delete the file to rebuild)")
else:
    REPO = "NTU-NLP-sg/xCodeEval"
    random.seed(0)
    t0 = time.time()

    # List the code_compilation training shards (jsonl files) and visit them in random order.
    shards = sorted(f for f in list_repo_files(REPO, repo_type="dataset")
                    if f.startswith("code_compilation/train/") and f.endswith(".jsonl"))
    print(len(shards), "train shards, e.g.", shards[:2])
    random.shuffle(shards)

    pos, neg = [], []     # pos = compiles (label 1), neg = compilation error (label 0)
    for shard in shards[:XC_MAX_SHARDS]:
        path = hf_hub_download(REPO, shard, repo_type="dataset")
        with open(path, encoding="utf-8") as f:
            for line in f:
                ex = json.loads(line)
                # keep Python 3 / PyPy 3 only (Python 2 code would "fail to compile" for the wrong reason)
                if ex["lang_cluster"] != XC_LANG_CLUSTER or "3" not in ex["lang"]:
                    continue
                code = ex["source_code"]
                if len(code) > 1500:                                    # cheap character-length pre-filter
                    continue
                if len(enc.encode_ordinary(code)) > BLOCK_SIZE:        # whole program must fit in the context window
                    continue
                label = 0 if ex["compilation_error"] else 1
                bucket = pos if label == 1 else neg
                if len(bucket) < XC_N_PER_CLASS:
                    # "group" = problem id; used later to keep the same problem out of both train and test
                    bucket.append({"code": code, "label": label, "group": ex["src_uid"]})
        print(f"{shard}: pos={len(pos)} neg={len(neg)}  ({(time.time()-t0)/60:.1f} min)")
        if len(pos) >= XC_N_PER_CLASS and len(neg) >= XC_N_PER_CLASS:
            break

    json.dump(pos + neg, open(LABEL_FILE, "w", encoding="utf-8"))
    print("saved", len(pos) + len(neg), "examples;", len(pos), "compile /", len(neg), "error")
    # Compile errors are the rare class. If neg is much smaller than XC_N_PER_CLASS, raise XC_MAX_SHARDS:
    # class balancing later is limited by the smaller class.

labels_xcodeeval.json already exists - reusing it (delete the file to rebuild)


## 2. Pre-training data (CodeParrot)

In [4]:
import os, numpy as np, tiktoken
from datasets import load_dataset

enc = tiktoken.get_encoding("gpt2")     # GPT-2 BPE tokenizer (vocab 50,257); also used later for the labelled code
out = "data/codeparrot"                 # relative to the nanoGPT folder; train.py reads data/<dataset>/{train,val}.bin
os.makedirs(out, exist_ok=True)

def dump(name, split, max_files):
    # Stream `max_files` files from the HF dataset, tokenize each, and join them into one flat uint16 array,
    # with an end-of-text token between files. (nanoGPT's .bin format.)
    ds = load_dataset(name, split=split, streaming=True)
    chunks = []                          # one small numpy array per file (a python list of ints would use ~8 GB at this scale)
    for i, ex in enumerate(ds):
        if i >= max_files: break
        chunks.append(np.array(enc.encode_ordinary(ex["content"]) + [enc.eot_token], dtype=np.uint16))
    return np.concatenate(chunks)

train = dump("codeparrot/codeparrot-clean", "train", PRETRAIN_FILES)
val   = dump("codeparrot/codeparrot-clean-valid", "train", 500)   # held-out files for the pre-training val loss
train.tofile(os.path.join(out, "train.bin"))
val.tofile(os.path.join(out, "val.bin"))
print(f"{len(train):,} train tokens, {len(val):,} val tokens")
print(f"pre-training will make {tok_per_iter*PRETRAIN_ITERS/len(train):.2f} passes over the data")

Resolving data files:   0%|          | 0/54 [00:00<?, ?it/s]

Repo card metadata block was not found. Setting CardData to empty.


88,644,507 train tokens, 2,370,285 val tokens
pre-training will make 3.70 passes over the data


## 3. Pre-train

In [5]:
# Pre-trains one GPT per positional encoding in POS_ENCS, one after the other (language-model pre-training from scratch).
# Each checkpoint goes to pretrain_dir(pe)/ckpt.pt, which the classifier loads in section 4.
# --compile=False avoids torch.compile problems with the custom RoPE/position code; you can try =True on the GPU for speed.
# --warmup_iters: LR warm-up over the first 5% of iterations, then cosine decay to the end (lr_decay_iters = max_iters).
# Same data, model size, seed and schedule for every encoding, so only the positional encoding differs.
import gc
for pe in POS_ENCS:
    pre_dir = pretrain_dir(pe)
    if REUSE_PRETRAINED and os.path.exists(f"{pre_dir}/ckpt.pt"):
        print(f"[{pe}] checkpoint exists, skipping pre-training"); continue
    print(f"\n===== pre-training: pos_enc={pe} -> {pre_dir} =====")
    get_ipython().run_line_magic("run",
        f"train.py --dataset=codeparrot --out_dir={pre_dir} "
        f"--device={DEVICE} --compile=False --pos_enc={pe} --warmup_iters={PRETRAIN_ITERS // 20} "
        f"--block_size={BLOCK_SIZE} --batch_size={BATCH_SIZE} --gradient_accumulation_steps={GRAD_ACCUM} "
        f"--n_layer={N_LAYER} --n_head={N_HEAD} --n_embd={N_EMBD} --dropout=0.0 "
        f"--max_iters={PRETRAIN_ITERS} --lr_decay_iters={PRETRAIN_ITERS} "
        f"--eval_interval=250 --eval_iters=20 --log_interval=500")
    # %run leaves train.py's variables (model, optimizer, ...) in the notebook; free the GPU memory before the next run
    for name in ("model", "optimizer", "scaler", "raw_model", "X", "Y"):
        globals().pop(name, None)
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()


===== pre-training: pos_enc=rope -> out-code-rope-10000it-20000f =====
Overriding: dataset = codeparrot
Overriding: out_dir = out-code-rope-10000it-20000f
Overriding: device = cuda
Overriding: compile = False
Overriding: pos_enc = rope
Overriding: warmup_iters = 500
Overriding: block_size = 256
Overriding: batch_size = 64
Overriding: gradient_accumulation_steps = 2
Overriding: n_layer = 8
Overriding: n_head = 8
Overriding: n_embd = 512
Overriding: dropout = 0.0
Overriding: max_iters = 10000
Overriding: lr_decay_iters = 10000
Overriding: eval_interval = 250
Overriding: eval_iters = 20
Overriding: log_interval = 500
tokens per iteration will be: 32,768
Initializing a new model from scratch
defaulting to vocab_size of GPT-2 to 50304 (50257 rounded up for efficiency)
number of parameters: 50.93M


/home/oliver/DeepML-project/train.py:202: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))


num decayed parameter tensors: 33, with 50,921,472 parameters
num non-decayed parameter tensors: 17, with 8,704 parameters
using fused AdamW: True
step 0: train loss 10.3149, val loss 10.3178
iter 0: loss 10.3197, time 4688.12ms, mfu -100.00%
step 250: train loss 3.0201, val loss 3.1212
saving checkpoint to out-code-rope-10000it-20000f
step 500: train loss 2.4217, val loss 2.5521
saving checkpoint to out-code-rope-10000it-20000f
iter 500: loss 2.5239, time 8663.01ms, mfu 0.39%
step 750: train loss 1.9458, val loss 2.0755
saving checkpoint to out-code-rope-10000it-20000f
step 1000: train loss 1.7546, val loss 1.9030
saving checkpoint to out-code-rope-10000it-20000f
iter 1000: loss 1.7664, time 8629.07ms, mfu 0.39%
step 1250: train loss 1.6228, val loss 1.7480
saving checkpoint to out-code-rope-10000it-20000f
step 1500: train loss 1.5953, val loss 1.7600
saving checkpoint to out-code-rope-10000it-20000f
iter 1500: loss 1.7661, time 8631.17ms, mfu 0.39%
step 1750: train loss 1.5348, val l

/home/oliver/DeepML-project/train.py:202: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=(dtype == 'float16'))


step 0: train loss 10.6452, val loss 10.6395
iter 0: loss 10.6338, time 3881.44ms, mfu -100.00%
step 250: train loss 3.2696, val loss 3.2477
saving checkpoint to out-code-abs-10000it-20000f
step 500: train loss 2.5092, val loss 2.6483
saving checkpoint to out-code-abs-10000it-20000f
iter 500: loss 2.5157, time 8369.98ms, mfu 0.40%
step 750: train loss 2.0921, val loss 2.2356
saving checkpoint to out-code-abs-10000it-20000f
step 1000: train loss 1.8605, val loss 1.9990
saving checkpoint to out-code-abs-10000it-20000f
iter 1000: loss 1.7883, time 8379.54ms, mfu 0.40%
step 1250: train loss 1.7399, val loss 1.8917
saving checkpoint to out-code-abs-10000it-20000f
step 1500: train loss 1.6518, val loss 1.8301
saving checkpoint to out-code-abs-10000it-20000f
iter 1500: loss 1.6693, time 8366.85ms, mfu 0.40%
step 1750: train loss 1.6136, val loss 1.7831
saving checkpoint to out-code-abs-10000it-20000f
step 2000: train loss 1.5811, val loss 1.7553
saving checkpoint to out-code-abs-10000it-20000

## 4. Classifier model (configurable head depth, partial freezing)

In [6]:
import contextlib, torch, torch.nn as nn
from model import GPT, GPTConfig

BLOCK = BLOCK_SIZE

def use_pos_enc(pe):
    # Switch everything that depends on the positional encoding: the checkpoint build_gpt() loads
    # (weights + model hyper-parameters written by train.py) and the results file the grid writes to.
    global POS_ENC, PRETRAIN_DIR, RESULTS_CSV, ckpt
    POS_ENC, PRETRAIN_DIR = pe, pretrain_dir(pe)
    RESULTS_CSV = f"grid_results_{pe}.csv"
    ckpt = torch.load(f"{PRETRAIN_DIR}/ckpt.pt", map_location="cpu", weights_only=False)

use_pos_enc(POS_ENC)      # start with the first encoding; the grid loop switches through all of them

def build_gpt(pretrained=True):
    # Same architecture as the checkpoint; load the pre-trained weights, or leave them random for the scratch baseline.
    gpt = GPT(GPTConfig(**ckpt["model_args"]))
    if pretrained:
        sd = {k.replace("_orig_mod.", ""): v for k, v in ckpt["model"].items()}   # strip torch.compile prefix if present
        gpt.load_state_dict(sd)
    return gpt

def make_head(n_embd, depth, n_classes=2, hidden=256, p=0.1):
    # depth 1: Dropout -> Linear(n_embd, n_classes)                       (linear probe)
    # depth 2: Dropout -> Linear(n_embd, hidden) -> GELU -> Dropout -> Linear(hidden, n_classes)
    # depth 3: one more Linear(hidden, hidden) -> GELU -> Dropout block in the middle
    assert depth in (1, 2, 3)
    layers, d = [nn.Dropout(p)], n_embd
    for _ in range(depth - 1):
        layers += [nn.Linear(d, hidden), nn.GELU(), nn.Dropout(p)]
        d = hidden
    layers.append(nn.Linear(d, n_classes))
    return nn.Sequential(*layers)

class CodeClassifier(nn.Module):
    # GPT backbone + classification head.
    # n_freeze = number of bottom blocks to freeze (embeddings are frozen whenever n_freeze > 0;
    # n_freeze >= n_layer freezes the final LayerNorm too, i.e. the whole GPT).
    def __init__(self, gpt, head_depth=1, hidden=256, n_classes=2, n_freeze=0):
        super().__init__()
        self.gpt = gpt
        self.n_layer = len(gpt.transformer.h)
        self.n_freeze = min(n_freeze, self.n_layer)
        self.head = make_head(gpt.config.n_embd, head_depth, n_classes, hidden)

        # Switch off gradients for everything that should stay frozen.
        t, k = gpt.transformer, self.n_freeze
        frozen = []
        if k > 0:
            frozen += [t.wte] + ([t.wpe] if "wpe" in t else []) + list(t.h[:k])   # embeddings + bottom k blocks
        if k >= self.n_layer:
            frozen.append(t.ln_f)                                                  # fully frozen: final LayerNorm too
        for mod in frozen:
            for p in mod.parameters():
                p.requires_grad = False

    def forward(self, idx, mask):
        # idx: (batch, length) token ids, mask: (batch, length) 1 for real tokens, 0 for padding
        t, k = self.gpt.transformer, self.n_freeze
        no_grad = torch.no_grad() if k > 0 else contextlib.nullcontext()
        with no_grad:                                   # frozen bottom part: no backprop through it, so it is faster
            x = self.gpt.embed(idx)                     # wte (+ wpe for 'abs'); RoPE acts inside attention
            for block in t.h[:k]:
                x = block(x)
        for block in t.h[k:]:                           # trainable blocks
            x = block(x)
        if k >= self.n_layer:
            with torch.no_grad():
                x = t.ln_f(x)
        else:
            x = t.ln_f(x)
        # Mean-pool over the real (non-padding) tokens -> one vector per program -> head -> class logits
        m = mask.unsqueeze(-1).float()
        pooled = (x * m).sum(1) / m.sum(1)
        return self.head(pooled)

## 5. Data and training helpers

In [7]:
rows_all = json.load(open(LABEL_FILE, encoding="utf-8"))

# Split by problem (src_uid) so the same programming task never appears in both train and test.
uids = sorted({r["group"] for r in rows_all})
random.Random(0).shuffle(uids)
test_uids = set(uids[:int(0.2 * len(uids))])           # 20% of problems -> test set
test_rows  = [r for r in rows_all if r["group"] in test_uids]
train_pool = [r for r in rows_all if r["group"] not in test_uids]
random.Random(0).shuffle(train_pool)
print(f"pool: {len(train_pool)} train / {len(test_rows)} test")
n_pos_tr = sum(r["label"] == 1 for r in train_pool); n_neg_tr = len(train_pool) - n_pos_tr
MAX_BALANCED = 2 * min(n_pos_tr, n_neg_tr)
print(f"train pool: {n_pos_tr} compile / {n_neg_tr} error -> largest class-balanced training set = {MAX_BALANCED}")
n_pos_test = sum(r["label"] == 1 for r in test_rows)
n_neg_test = len(test_rows) - n_pos_test
print(f"test set: {n_pos_test} compile / {n_neg_test} error -> always guessing the majority class "
      f"scores {max(n_pos_test, n_neg_test)/len(test_rows):.3f} accuracy (balanced accuracy 0.500)")

def get_train_rows(n_labeled):
    # Class-balanced subset: n_labeled/2 programs per class, or fewer if compile errors (the rare class) run out.
    # (So the effective size can be smaller than n_labeled.) None = whole pool, NOT balanced.
    # Always the same first-k examples per class, so a smaller set is contained in a larger one.
    if n_labeled is None: return list(train_pool)
    pos = [r for r in train_pool if r["label"] == 1]
    neg = [r for r in train_pool if r["label"] == 0]
    k = min(n_labeled // 2, len(pos), len(neg))
    return pos[:k] + neg[:k]

def make_batch(rows):
    # Tokenize a list of examples, truncate to BLOCK tokens, right-pad to the longest in the batch.
    ids = [enc.encode_ordinary(r["code"])[:BLOCK] for r in rows]
    L = max(len(i) for i in ids)
    x = torch.zeros(len(ids), L, dtype=torch.long)
    m = torch.zeros(len(ids), L, dtype=torch.long)
    for k, i in enumerate(ids):
        x[k, :len(i)] = torch.tensor(i); m[k, :len(i)] = 1     # right-padded; causal attention => real tokens never see padding
    y = torch.tensor([r["label"] for r in rows])
    return x.to(DEVICE), m.to(DEVICE), y.to(DEVICE)

def evaluate(model, rows):
    # Returns (accuracy, balanced accuracy). Balanced accuracy = mean of the per-class recalls, so a classifier
    # that always predicts the majority class scores 0.5 even when the test set is imbalanced.
    model.eval(); hit = {0: 0, 1: 0}; tot = {0: 0, 1: 0}
    with torch.no_grad():
        for i in range(0, len(rows), 64):
            x, m, y = make_batch(rows[i:i+64])
            pred = model(x, m).argmax(-1)
            for c in (0, 1):
                sel = y == c
                tot[c] += sel.sum().item(); hit[c] += (pred[sel] == c).sum().item()
    model.train()
    acc = (hit[0] + hit[1]) / len(rows)
    recalls = [hit[c] / tot[c] for c in (0, 1) if tot[c] > 0]
    return acc, sum(recalls) / len(recalls)

def train_classifier(pretrained=True, head_depth=None, n_labeled=None, epochs=None,
                     n_freeze=None, bs=16, seed=0, verbose=True):
    # Train one classifier and return (model, test accuracy per epoch, test balanced accuracy per epoch).
    # Any argument left as None falls back to the config cell.
    # pretrained=False -> random-init backbone (baseline). seed changes head init, dropout and data order
    # (the labelled subset itself is fixed).
    head_depth = HEAD_DEPTH if head_depth is None else head_depth
    n_labeled  = N_LABELED if n_labeled is None else n_labeled
    epochs     = FT_EPOCHS if epochs is None else epochs
    n_freeze   = N_FREEZE if n_freeze is None else n_freeze

    torch.manual_seed(seed); random.seed(seed)
    rows = get_train_rows(n_labeled)
    model = CodeClassifier(build_gpt(pretrained), head_depth, HEAD_HIDDEN, n_freeze=n_freeze).to(DEVICE)
    n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"pretrained={pretrained} depth={head_depth} freeze={n_freeze}/{model.n_layer} "
          f"train={len(rows)} seed={seed} trainable={n_tr/1e6:.2f}M")

    # Two learning rates: the new head learns fast, the (unfrozen) GPT blocks change slowly.
    gpt_params = [p for p in model.gpt.parameters() if p.requires_grad]   # empty when fully frozen
    groups_ = [{"params": model.head.parameters(), "lr": LR_HEAD}]
    if gpt_params:
        groups_.append({"params": gpt_params, "lr": LR_BACKBONE if pretrained else LR_BACKBONE_SCRATCH})
    opt = torch.optim.AdamW(groups_, weight_decay=0.01)
    loss_fn = nn.CrossEntropyLoss()

    accs, bals = [], []
    model.train()
    for epoch in range(epochs):
        random.shuffle(rows)
        for step, i in enumerate(range(0, len(rows), bs)):
            x, m, y = make_batch(rows[i:i+bs])
            loss = loss_fn(model(x, m), y)
            opt.zero_grad(); loss.backward(); opt.step()
            if verbose and step % 50 == 0:
                print(f"  epoch {epoch} step {step} loss {loss.item():.3f}")
        a_, b_ = evaluate(model, test_rows); accs.append(a_); bals.append(b_)
        if verbose:
            print(f"epoch {epoch}: test acc {accs[-1]:.3f} (balanced {bals[-1]:.3f})")
    return model, accs, bals

pool: 8037 train / 1963 test
train pool: 4237 compile / 3800 error -> largest class-balanced training set = 7600
test set: 763 compile / 1200 error -> always guessing the majority class scores 0.611 accuracy (balanced accuracy 0.500)


## 6. Single runs

In [8]:
# Two runs using the settings in the config cell:
model_pre, accs_pre, bals_pre = train_classifier(pretrained=True)                       # pre-trained GPT + head
model_scratch, accs_scratch, bals_scratch = train_classifier(pretrained=False, n_freeze=0)  # random-init baseline (same head and data)

number of parameters: 50.93M
pretrained=True depth=1 freeze=0/8 train=4800 seed=0 trainable=50.93M
  epoch 0 step 0 loss 0.823
  epoch 0 step 50 loss 0.891
  epoch 0 step 100 loss 0.499
  epoch 0 step 150 loss 0.761
  epoch 0 step 200 loss 0.275
  epoch 0 step 250 loss 0.498
epoch 0: test acc 0.616 (balanced 0.619)
  epoch 1 step 0 loss 0.270
  epoch 1 step 50 loss 0.431
  epoch 1 step 100 loss 0.142
  epoch 1 step 150 loss 0.162
  epoch 1 step 200 loss 0.240
  epoch 1 step 250 loss 0.034
epoch 1: test acc 0.627 (balanced 0.613)
  epoch 2 step 0 loss 0.261
  epoch 2 step 50 loss 0.280
  epoch 2 step 100 loss 0.095
  epoch 2 step 150 loss 0.034
  epoch 2 step 200 loss 0.049
  epoch 2 step 250 loss 0.100
epoch 2: test acc 0.640 (balanced 0.600)
  epoch 3 step 0 loss 0.045
  epoch 3 step 50 loss 0.254
  epoch 3 step 100 loss 0.007
  epoch 3 step 150 loss 0.036
  epoch 3 step 200 loss 0.014
  epoch 3 step 250 loss 0.039
epoch 3: test acc 0.657 (balanced 0.655)
  epoch 4 step 0 loss 0.021
 

## 7. Experiment grid: initialisation x frozen blocks x head depth x labelled-data size x seed

In [9]:
import pandas as pd, time

def run_grid():
    # Full grid (see the "What is being compared" table). Seeds are the outer loop, so if you stop early you
    # still have complete grids for the first seeds. Each finished run is appended to RESULTS_CSV.
    # Labelled sizes are capped by the number of compile-error examples (balanced sets); duplicates are merged.
    grid_sizes = sorted({min(n, MAX_BALANCED) for n in GRID_SIZES})
    print("effective labelled sizes:", grid_sizes)
    jobs = []
    for seed in GRID_SEEDS:
        for k in GRID_FREEZE:
            for depth in GRID_DEPTHS:
                for n in grid_sizes:
                    jobs.append(dict(mode=f"pretrained, freeze {k}/{N_LAYER}", pretrained=True,
                                     n_freeze=k, depth=depth, n_labeled=n, seed=seed))
        for depth in GRID_DEPTHS:                      # random-init baseline, fully trainable
            for n in grid_sizes:
                jobs.append(dict(mode="scratch", pretrained=False,
                                 n_freeze=0, depth=depth, n_labeled=n, seed=seed))
    print(len(jobs), "runs")

    results, t0 = [], time.time()
    for j, job in enumerate(jobs, 1):
        _, a, b = train_classifier(pretrained=job["pretrained"], n_freeze=job["n_freeze"], head_depth=job["depth"],
                                n_labeled=job["n_labeled"], epochs=GRID_EPOCHS, seed=job["seed"], verbose=False)
        results.append(dict(pos_enc=POS_ENC, mode=job["mode"], depth=job["depth"], n_labeled=job["n_labeled"],
                            seed=job["seed"], final_acc=a[-1], final_bal_acc=b[-1]))     # accuracy after the LAST epoch (no best-epoch picking on the test set)
        pd.DataFrame(results).to_csv(RESULTS_CSV, index=False)
        print(f"[{j}/{len(jobs)}] acc={a[-1]:.3f} bal_acc={b[-1]:.3f}  elapsed {(time.time()-t0)/60:.1f} min")

for pe in POS_ENCS:                 # one full grid per positional encoding (each writes grid_results_<pe>.csv)
    use_pos_enc(pe)
    print(f"\n===== fine-tuning grid: pos_enc={pe} =====")
    run_grid()


===== fine-tuning grid: pos_enc=rope =====
effective labelled sizes: [500, 1000, 2500, 5000]
180 runs
number of parameters: 50.93M
pretrained=True depth=1 freeze=0/8 train=500 seed=0 trainable=50.93M
[1/180] acc=0.558 bal_acc=0.608  elapsed 0.4 min
number of parameters: 50.93M
pretrained=True depth=1 freeze=0/8 train=1000 seed=0 trainable=50.93M
[2/180] acc=0.621 bal_acc=0.598  elapsed 0.9 min
number of parameters: 50.93M
pretrained=True depth=1 freeze=0/8 train=2500 seed=0 trainable=50.93M
[3/180] acc=0.651 bal_acc=0.621  elapsed 2.0 min
number of parameters: 50.93M
pretrained=True depth=1 freeze=0/8 train=5000 seed=0 trainable=50.93M
[4/180] acc=0.632 bal_acc=0.600  elapsed 3.8 min
number of parameters: 50.93M
pretrained=True depth=2 freeze=0/8 train=500 seed=0 trainable=51.06M
[5/180] acc=0.596 bal_acc=0.620  elapsed 4.2 min
number of parameters: 50.93M
pretrained=True depth=2 freeze=0/8 train=1000 seed=0 trainable=51.06M
[6/180] acc=0.607 bal_acc=0.587  elapsed 4.7 min
number of p

In [10]:
# Summary tables: rows = (positional encoding, mode, head depth), columns = number of labelled examples.
# Reads every grid_results_*.csv present, so after running both rope and abs you get the comparison here.
# Balanced accuracy is the one to trust if the test set is imbalanced (see the "test set:" line above).
# Differences of ~1-2 points are within noise with this test-set size; check the std across seeds.
import glob
df = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("grid_results_*.csv"))], ignore_index=True)
tab = lambda col, f: df.pivot_table(index=["pos_enc", "mode", "depth"], columns="n_labeled", values=col, aggfunc=f)
print("mean BALANCED accuracy over seeds"); print(tab("final_bal_acc", "mean").round(3))
print("\nstd of balanced accuracy over seeds"); print(tab("final_bal_acc", "std").round(3))
print("\nmean plain accuracy over seeds"); print(tab("final_acc", "mean").round(3))

mean BALANCED accuracy over seeds
n_labeled                              500    1000   2500   5000
pos_enc mode                   depth                            
abs     pretrained, freeze 0/8 1      0.621  0.598  0.605  0.621
                               2      0.584  0.608  0.610  0.618
                               3      0.601  0.581  0.580  0.610
        pretrained, freeze 2/8 1      0.628  0.598  0.621  0.600
                               2      0.601  0.565  0.591  0.622
                               3      0.600  0.592  0.597  0.599
        pretrained, freeze 4/8 1      0.622  0.619  0.603  0.640
                               2      0.614  0.612  0.620  0.634
                               3      0.616  0.606  0.630  0.624
        pretrained, freeze 8/8 1      0.639  0.593  0.612  0.592
                               2      0.605  0.581  0.610  0.609
                               3      0.606  0.595  0.606  0.620
        scratch                1      0.578  0.594  0.58